In [ ]:
using Pkg

cd(@__DIR__)
Pkg.activate("../")
using Dates, SonifSismo
using WAV
using DSP
using Statistics
using Interpolations

Pkg.update("CairoMakie")   # mettre à jour
using CairoMakie


include("C:/Users/ameli/Desktop/Stage/instruments.jl/instruments_v2.jl")

archive = DataArchive("C:/Users/ameli/Desktop/Stage/mtFujiContinuous")


  Activating project at `c:\Users\ameli\Desktop\Stage\sonifSismo.jl`
    Updating registry at `C:\Users\ameli\.julia\registries\General.toml`
     Project No packages added to or removed from `C:\Users\ameli\Desktop\Stage\sonifSismo.jl\Project.toml`
    Manifest No packages added to or removed from `C:\Users\ameli\Desktop\Stage\sonifSismo.jl\Manifest.toml`


LoadError: LoadError: ArgumentError: invalid JSON at byte position 1 while parsing type Any: InvalidChar
parameters.json

in expression starting at C:\Users\ameli\Desktop\Stage\instruments.jl\instruments_v2.jl:5

In [2]:
t0_utc = DateTime(2008, 5, 25, 9, 30, 0) 
t1_utc = DateTime(2008, 5, 25, 12, 0, 0) 


events = read_catalog(archive; start=t0_utc, stop=t1_utc)


traces = read_window(
    archive,  t0_utc, t1_utc;
    stations  = "MTS2",
    channels  = ["wE"],
    processed = true,
    bandpass  = (1, 15.0),
)

include("C:/Users/ameli/Desktop/Stage/sonifSismo.jl/examples/sonify_one_trace.jl")


trace = only(t for t in traces if strip(t.sta.cha) == "wE")


wavefig = plot_waveforms(
    traces;
    title  = "MTS2",
)

wavefig


UndefVarError: UndefVarError: `archive` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [3]:
fs  = 44100
raw = Float64.(Seis.trace(trace))

sig_bg  = filtfilt(digitalfilter(Lowpass(1.0;        fs=fs), Butterworth(4)), raw)
sig_lfe = filtfilt(digitalfilter(Bandpass(1.0, 5.0;  fs=fs), Butterworth(4)), raw)
sig_eq  = filtfilt(digitalfilter(Bandpass(5.0, 15.0; fs=fs), Butterworth(4)), raw)

n       = length(raw)
hz_base = 196.0
duree   = n / fs

UndefVarError: UndefVarError: `Seis` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: Seis is loaded but not imported in the active module Main.

In [4]:

function make_instrument_sound(instrument_name, base_hz, seconds)
    params = deepcopy(DRUM_PRESETS[instrument_name])

    if haskey(params, "frequencies")
        params["frequencies"] = [base_hz * m for m in params["frequencies"]]
    end

    params["sustain_time"] = seconds*0.85
    params["attack"] = seconds*0.05
    params["release"] = seconds*0.10

    return synthesize_drum(params; seconds=seconds, fs=44100.0)
end

make_instrument_sound (generic function with 1 method)

In [5]:
son_bg  = make_instrument_sound("Ethereal Pad", hz_base,     duree) .* (sig_bg  ./ (maximum(abs.(sig_bg))  + 1e-9))
son_lfe = make_instrument_sound("Violin",       hz_base * 2, duree) .* (sig_lfe ./ (maximum(abs.(sig_lfe)) + 1e-9))
son_eq  = make_instrument_sound("Glass Harp",   hz_base * 4, duree) .* (sig_eq  ./ (maximum(abs.(sig_eq))  + 1e-9))

UndefVarError: UndefVarError: `hz_base` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [6]:
dossier = "C:/Users/ameli/Desktop/Stage/Résultats/tentative_instruments"
mkpath(dossier)

mixed = 0.6 .* son_bg .+ 0.8 .* son_lfe .+ 0.6 .* son_eq

wavwrite(Float32.(mixed), joinpath(dossier, "musique_instruments.wav"); Fs=fs)
println("Fichier écrit.")

UndefVarError: UndefVarError: `son_bg` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.